# Vacancy formation energy in bcc Fe (EAM)

Builds a Fe supercell from `bulk('Fe', a=2.85, cubic=True)`, removes
one atom, and computes `E_vac - E_bulk` (the bare difference) under
the included Al-Fe.eam.fs EAM potential.

Driven from a `Workflow` so that `engine.with_working_directory(...)`
is evaluated eagerly with the resolved engine value.


In [1]:
from ase.build import bulk
from ase.calculators.eam import EAM

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize, run
from pyiron_workflow_atomistics.structure import (
    create_supercell_with_min_dimensions,
    create_vacancy,
)
from pyiron_workflow_atomistics.physics.point_defect import (
    calculate_vacancy_formation_energy,
)

engine = ASEEngine(
    EngineInput=CalcInputMinimize(
        force_convergence_tolerance=0.05, max_iterations=100
    ),
    calculator=EAM(potential="Al-Fe.eam.fs"),
    working_directory="./_vac_runs",
)


In [2]:
wf = Workflow("vacancy_formation_energy")
wf.supercell = create_supercell_with_min_dimensions(
    bulk("Fe", a=2.85, cubic=True), min_dimensions=[7, 7, 7]
)
wf.with_vacancy = create_vacancy(wf.supercell, remove_atom_index=0)
wf.calc_bulk = run(
    wf.supercell, engine=engine.with_working_directory("supercell"), label="calc_bulk"
)
wf.calc_vac = run(
    wf.with_vacancy, engine=engine.with_working_directory("vacancy"), label="calc_vac"
)
wf.E_form = calculate_vacancy_formation_energy(
    vacancy_energy=wf.calc_vac.outputs.engine_output.final_energy,
    supercell_energy=wf.calc_bulk.outputs.engine_output.final_energy,
)
wf.run()

e_f = wf.E_form.outputs.formation_energy.value
print(f"Vacancy formation energy (E_vac - E_bulk) of Fe in BCC Fe "
      f"(Al-Fe.eam.fs): {e_f:.3f} eV")


      Step     Time          Energy          fmax
BFGS:    0 14:53:19     -216.690138        0.000000


      Step     Time          Energy          fmax
BFGS:    0 14:53:20     -210.847774        0.368379


BFGS:    1 14:53:21     -210.864488        0.330939


BFGS:    2 14:53:21     -210.942616        0.131664


BFGS:    3 14:53:22     -210.947698        0.119259


BFGS:    4 14:53:23     -210.961581        0.038241


Vacancy formation energy (E_vac - E_bulk) of Fe in BCC Fe (Al-Fe.eam.fs): 5.729 eV
